# PAI102: Data Pipeline to Monitor Local Crime Trends

This lab notebook uses the supplied instructor notebook's core workflow: Socrata/CSV extraction, data validation, duplicate removal and time features, PostgreSQL loading, and Prefect scheduling. The runnable source is in `src/crime_pipeline/`; the dashboard is `dashboard.py`.

The notebook uses the supplied CSV for a repeatable lab run. The project can switch to the live Cambridge Socrata API for scheduled cloud refreshes.


## Pipeline design

1. **Extract:** read the supplied police log CSV or fetch Socrata dataset `3gki-wyrb`.
2. **Validate:** map CSV and API column names to one schema, parse source dates, and report invalid required fields and duplicate IDs.
3. **Transform:** remove invalid rows and duplicate IDs, then add year, month, day, hour, minute, second, month name, and weekday.
4. **Load:** replace the PostgreSQL snapshot in a transaction.
5. **Orchestrate:** Prefect tracks the tasks and schedules daily refreshes.
6. **Visualize:** Streamlit reads aggregate SQL results and displays report counts.

The supplied notebook's original Metabase plan is represented here by Streamlit, which the notebook allowed as an alternative. This keeps the dashboard ready to open as soon as the project is started.


## Before running this notebook

- Install and start Docker Desktop.
- Copy `Daily_Police_Log_20261106.csv` into the project's `data` folder as `police_log.csv`.
- Copy `.env.example` to `.env` and replace the example PostgreSQL password.
- In a terminal opened in the project folder, start services and load the CSV once:

~~~powershell
docker compose up -d --build
docker compose run --rm -e RUN_MODE=once pipeline
~~~

- Install Python packages into the notebook kernel from the project folder by running the next cell.


In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src" / "crime_pipeline").exists():
    raise RuntimeError("Open this notebook with the project folder as the working directory.")
print(f"Project folder: {project_root}")

## Extract and validate

The extractor reads the `CSV_PATH` in `.env`. The validator accepts both the uploaded CSV's display headers and the lowercase Socrata API fields. Blank subtype and description values are allowed because those fields are optional.


In [ ]:
from crime_pipeline.extract import extract_data
from crime_pipeline.validate import validate_data

raw_data = extract_data.fn()
validated_data, quality_report = validate_data.fn(raw_data)

print(f"Source rows: {quality_report['source_rows']:,}")
print(f"Valid rows: {quality_report['valid_rows']:,}")
print(f"Invalid required-field rows removed: {quality_report['invalid_required_rows_removed']:,}")
print(f"Duplicate IDs found: {quality_report['duplicate_ids_found']:,}")
print(f"Blank subtype values: {quality_report['blank_subtype_rows']:,}")
print(f"Blank descriptions: {quality_report['blank_description_rows']:,}")

## Transform the records

The same ID-based deduplication and calendar splitting described in the instructor notebook happen here. The new fields make monthly and time-of-day analysis straightforward in SQL.


In [ ]:
from crime_pipeline.transform import transform_data

transformed_data = transform_data.fn(validated_data)
print(f"Transformed rows: {len(transformed_data):,}")
display(transformed_data[["id", "date_time", "type", "year", "month", "day", "hour", "weekday"]].head())

## Load PostgreSQL

The PostgreSQL container must be running. From a notebook running on Windows, `.env` uses `localhost:5433`, which is forwarded to the database container. Each load replaces the current snapshot atomically, so rerunning the cell does not add duplicate copies.


In [ ]:
from crime_pipeline.load import load_into_postgres

rows_loaded = load_into_postgres.fn(transformed_data)
print(f"Rows loaded: {rows_loaded:,}")

## Review an aggregate query

This query matches the dashboard's monthly trend view. It does not retrieve report descriptions, incident IDs, or street locations.


In [ ]:
from sqlalchemy import create_engine, text
from crime_pipeline.config import database_url

engine = create_engine(database_url())
with engine.connect() as connection:
    monthly_counts = pd.read_sql_query(
        text("""
            SELECT date_trunc('month', date_time)::date AS month,
                   COUNT(*) AS reported_incidents
            FROM cpd_incidents
            GROUP BY 1
            ORDER BY 1
        """),
        connection,
    )
engine.dispose()
display(monthly_counts.tail(12))

## Open the dashboard and Prefect

- Dashboard: `http://localhost:8501`
- Prefect flow runs: `http://localhost:4200`

The project scheduler runs at midnight in `America/New_York`. After the initial CSV lab run, set `DATA_SOURCE=api` in `.env` and recreate the pipeline service to switch to live Socrata data. The full cloud deployment guide and five-parameter evaluation checklist are in `README.md`.

## Data interpretation

The dashboard counts reports recorded in the Cambridge Police Log. It does not establish that a report is a verified crime, and it does not calculate population-adjusted rates. The dashboard intentionally omits free-text descriptions, incident IDs, and street locations.
